# Level 1 — Task 3: Cleaning Data

**Objective:** Turn a deliberately messy customer file into an analysis-ready dataset. Every decision is documented.

**Tech stack:** Python, pandas, numpy

## 1. Data quality report

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW = Path("..") / "data" / "raw" / "messy_customer_data.csv"
OUT = Path("..") / "data" / "cleaned"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW)
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nNulls per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nNumeric describe:\n", df.describe(include="all").T.head(20))
df.head(10)


Shape: (855, 10)

Dtypes:
 customer_id          str
full_name            str
age              float64
gender               str
city                 str
signup_date          str
annual_income    float64
monthly_spend    float64
is_subscribed        str
score              int64
dtype: object

Nulls per column:
 customer_id        0
full_name          0
age               21
gender            88
city              59
signup_date      135
annual_income     28
monthly_spend     18
is_subscribed      0
score              0
dtype: int64

Duplicate rows: 40

Numeric describe:
                count unique         top freq          mean           std  \
customer_id      855    800       U1597    3           NaN           NaN   
full_name        855    815     User 13    2           NaN           NaN   
age            834.0    NaN         NaN  NaN     41.992806     15.443424   
gender           767      8        Male  176           NaN           NaN   
city             796      8       Delhi  154  

,customer_id,full_name,age,gender,city,signup_date,annual_income,monthly_spend,is_subscribed,score
0,U1000,User 0,37.0,female,Delhi,09/03/2024,54823.61,390.40,1,56
1,U1001,User 1,34.0,male,DELHI,07-10-2024,44162.12,122.16,No,95
2,U1002,User 2,31.0,Female,Bengaluru,NaN,88301.67,241.41,1,5
3,U1003,User 3,45.0,Male,Bangalore,2024-07-24,87393.97,43.05,0,84
4,U1004,User 4,47.0,male,Delhi,02-13-2024,55501.66,117.46,True,42
5,U1005,User 5,58.0,NaN,Mumbai,NaN,62134.15,144.86,1,44
6,U1006,User 6,49.0,Female,Bengaluru,2024-05-07,58088.45,124.79,YES,69
7,U1007,User 7,34.0,F,Hyderabad,10-09-2024,55960.21,NaN,False,77
8,U1008,User 8,43.0,M,Delhi,09/03/2024,56736.29,91.16,False,17
9,U1009,User 9,40.0,Male,DELHI,03-29-2024,76129.98,233.84,YES,71


In [2]:
def quality_report(frame, label):
    num = frame.select_dtypes(include=[np.number])
    anomalies = {}
    for col in num.columns:
        anomalies[col] = {
            "min": num[col].min(),
            "max": num[col].max(),
            "negatives": int((num[col] < 0).sum()) if pd.api.types.is_numeric_dtype(num[col]) else None,
        }
    return {
        "label": label,
        "rows": len(frame),
        "nulls": int(frame.isnull().sum().sum()),
        "duplicate_rows": int(frame.duplicated().sum()),
        "columns": list(frame.columns),
        "dtypes": {c: str(t) for c, t in frame.dtypes.items()},
        "range_anomalies": anomalies,
    }

before = quality_report(df, "before")
before


{'label': 'before',
 'rows': 855,
 'nulls': 349,
 'duplicate_rows': 40,
 'columns': ['customer_id',
  'full_name',
  'age',
  'gender',
  'city',
  'signup_date',
  'annual_income',
  'monthly_spend',
  'is_subscribed',
  'score'],
 'dtypes': {'customer_id': 'str',
  'full_name': 'str',
  'age': 'float64',
  'gender': 'str',
  'city': 'str',
  'signup_date': 'str',
  'annual_income': 'float64',
  'monthly_spend': 'float64',
  'is_subscribed': 'str',
  'score': 'int64'},
 'range_anomalies': {'age': {'min': np.float64(18.0),
   'max': np.float64(140.0),
   'negatives': 0},
  'annual_income': {'min': np.float64(2097.61),
   'max': np.float64(250000.0),
   'negatives': 0},
  'monthly_spend': {'min': np.float64(-80.0),
   'max': np.float64(530.43),
   'negatives': 6},
  'score': {'min': np.int64(1), 'max': np.int64(99), 'negatives': 0}}}

**Observation:** Nulls appear in age, gender, city, signup_date, income, and spend. Gender and city labels are inconsistent. Dates use mixed formats. `score` is object because of text like `N/A`. Duplicates exist. Age/income/spend contain impossible values (outliers).

## 2. Missing data handling

In [3]:
work = df.copy()

# Age: median imputation (skewed / outliers make mean a poor choice)
age_median = work["age"].median()
work["age"] = work["age"].fillna(age_median)

# Gender / city: unknown category (not a number — do not invent Male/Female)
work["gender"] = work["gender"].fillna("Unknown")
work["city"] = work["city"].fillna("Unknown")

# Income / spend: median (robust to outliers)
work["annual_income"] = work["annual_income"].fillna(work["annual_income"].median())
work["monthly_spend"] = work["monthly_spend"].fillna(work["monthly_spend"].median())

print("Nulls after imputation (except dates):")
print(work.isnull().sum())


Nulls after imputation (except dates):
customer_id        0
full_name          0
age                0
gender             0
city               0
signup_date      135
annual_income      0
monthly_spend      0
is_subscribed      0
score              0
dtype: int64


**Justification:**  
- **Median** for age/income/spend — these columns have extreme outliers, so the mean would be distorted.  
- **`Unknown` category** for gender/city — imputing the mode would fake demographic facts.  
- **Dates** are parsed next; remaining invalid dates are dropped because a fake signup date would break tenure analysis.

## 3. Duplicate removal

In [4]:
dup_count = int(work.duplicated().sum())
work = work.drop_duplicates()
print("Removed exact duplicate rows:", dup_count)
print("Rows now:", len(work))


Removed exact duplicate rows: 40
Rows now: 815


## 4. Standardization (gender, city, dates, flags)

In [5]:
def norm_gender(x):
    if pd.isna(x):
        return "Unknown"
    s = str(x).strip().lower()
    if s in {"m", "male"}:
        return "Male"
    if s in {"f", "female"}:
        return "Female"
    if s in {"other"}:
        return "Other"
    if s in {"unknown", "none", "nan"}:
        return "Unknown"
    return "Unknown"

def norm_city(x):
    s = str(x).strip().lower()
    mapping = {
        "mumbai": "Mumbai",
        "delhi": "Delhi",
        "bengaluru": "Bengaluru",
        "bangalore": "Bengaluru",
        "pune": "Pune",
        "hyderabad": "Hyderabad",
        "unknown": "Unknown",
    }
    return mapping.get(s, str(x).strip().title())

def parse_dates(series):
    a = pd.to_datetime(series, errors="coerce", format="mixed", dayfirst=True)
    return a

work["gender"] = work["gender"].map(norm_gender)
work["city"] = work["city"].map(norm_city)
work["signup_date"] = parse_dates(work["signup_date"])
print("Unparsed dates:", work["signup_date"].isna().sum())
# Drop rows with no usable signup date
before_drop = len(work)
work = work.dropna(subset=["signup_date"])
print("Dropped rows with invalid dates:", before_drop - len(work))

def norm_sub(x):
    s = str(x).strip().lower()
    if s in {"yes", "true", "1"}:
        return True
    if s in {"no", "false", "0"}:
        return False
    return pd.NA

work["is_subscribed"] = work["is_subscribed"].map(norm_sub).astype("boolean")
work["gender"].value_counts(dropna=False)


Unparsed dates: 129
Dropped rows with invalid dates: 129


gender
Female     296
Male       288
Unknown     71
Other       31
Name: count, dtype: int64

## 5. Outlier detection (IQR) — age, income, spend

In [6]:
def iqr_flags(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return lo, hi, (s < lo) | (s > hi)

decisions = {}
for col in ["age", "annual_income", "monthly_spend"]:
    lo, hi, mask = iqr_flags(work[col])
    n_out = int(mask.sum())
    # Cap (winsorize) instead of delete: keep customers, remove impossible influence
    work[col] = work[col].clip(lower=max(lo, 0) if col != "age" else max(lo, 18), upper=hi if col != "age" else min(hi, 90))
    if col == "age":
        work[col] = work[col].clip(18, 90)
    decisions[col] = {"iqr_low": round(lo, 2), "iqr_high": round(hi, 2), "flagged": n_out, "action": "cap/winsorize"}
pd.DataFrame(decisions).T


,iqr_low,iqr_high,flagged,action
age,-3.0,85.0,5,cap/winsorize
annual_income,6306.46,101481.31,17,cap/winsorize
monthly_spend,-50.16,344.74,30,cap/winsorize


**Decision:** Outliers are **capped**, not deleted. Removing rows would drop real customers who simply have unusual income/spend. Age is also clipped to 18–90 because values like 3 or 140 are data-entry errors.

## 6. Data type correction

In [7]:
work["customer_id"] = work["customer_id"].astype(str)
work["full_name"] = work["full_name"].astype(str).str.title()
work["age"] = work["age"].round().astype(int)
work["annual_income"] = work["annual_income"].astype(float)
work["monthly_spend"] = work["monthly_spend"].astype(float)
work["score"] = pd.to_numeric(work["score"].astype(str).str.strip(), errors="coerce")
work["score"] = work["score"].fillna(work["score"].median()).astype(int)
print(work.dtypes)
work.head()


customer_id                 str
full_name                   str
age                       int64
gender                      str
city                        str
signup_date      datetime64[us]
annual_income           float64
monthly_spend           float64
is_subscribed           boolean
score                     int64
dtype: object

,customer_id,full_name,age,gender,city,signup_date,annual_income,monthly_spend,is_subscribed,score
0,U1000,User 0,37,Female,Delhi,2024-03-09,54823.61,344.7375,True,56
1,U1001,User 1,34,Male,Delhi,2024-10-07,44162.12,122.1600,False,95
3,U1003,User 3,45,Male,Bengaluru,2024-07-24,87393.97,43.0500,False,84
4,U1004,User 4,47,Male,Delhi,2024-02-13,55501.66,117.4600,True,42
6,U1006,User 6,49,Female,Bengaluru,2024-07-05,58088.45,124.7900,True,69


## 7. Before vs after summary + save

In [8]:
after = quality_report(work, "after")
summary = pd.DataFrame({
    "metric": ["row_count", "null_cells", "duplicate_rows"],
    "before": [before["rows"], before["nulls"], before["duplicate_rows"]],
    "after": [after["rows"], after["nulls"], after["duplicate_rows"]],
})
print(summary)
print("\nDtype accuracy after clean:")
for c, t in after["dtypes"].items():
    print(f"  {c}: {t}")

clean_path = OUT / "cleaned_customer_data.csv"
work.to_csv(clean_path, index=False)
print("\nSaved:", clean_path.resolve())
summary


           metric  before  after
0       row_count     855    686
1      null_cells     349      0
2  duplicate_rows      40     13

Dtype accuracy after clean:
  customer_id: str
  full_name: str
  age: int64
  gender: str
  city: str
  signup_date: datetime64[us]
  annual_income: float64
  monthly_spend: float64
  is_subscribed: boolean
  score: int64

Saved: C:\Users\thris\Desktop\oasis-infobyte-data-analytics\data\cleaned\cleaned_customer_data.csv


,metric,before,after
0,row_count,855,686
1,null_cells,349,0
2,duplicate_rows,40,13


**Summary:** The file is now analysis-ready: consistent gender/city labels, datetime signup dates, numeric money/age/score, no exact duplicates, and documented treatment of missing values and outliers. The cleaned CSV is in `data/cleaned/cleaned_customer_data.csv`.